Scenario BL:
- Run 4 scenarios, kmeans, 12k, centroid
- Clean hydrogen only (no reformers, ammonia)
- No CCS
- Min/Max on all vRES, BESS, Electric Boilers, and infrastructure
- No SMR

In [ ]:
SCENARIOS = "base_run_EV"
#SCENARIOS = "base_run_GB"
#SCENARIOS = "base_run_HA" 
#SCENARIOS = "base_run_KM"

%run ~/Thesis/Research_Runs/Runfiles/Scenario_BL.py

KeyboardInterrupt: 

Analysis

In [1]:
from pathlib import Path
import sys


#SCENARIOS = "base_run_EV"
#SCENARIOS = "base_run_GB"
#SCENARIOS = "base_run_HA" 
SCENARIOS = "base_run_KM"
TIMESTAMP = "1007_2150"



###########################################################################
NAME = "BL_KM_10K" + SCENARIOS + "_2025" + TIMESTAMP
netcdf_filename = "output/1.Baseline/" + NAME + ".nc"
yaml_filename = "output/1.Baseline/" + NAME + ".yaml"

flow_csv_filename = "analysis_results/1.Baseline/flow_with_utilization_and_coords.csv"
analysis_html_filename = "analysis_results/1.Baseline/flows_costs_util.html"
geojson_filename = "Research_Runs/Postprocessing/NUTS2.geojson"
map_output_html = "analysis_results/1.Baseline/carrier_flows.html"
output_html_path = "analysis_results/1.Baseline/Installed_Capacities.html"

html_title = "Installed Capacities - BL KM"  + NAME
map_title = "Carrier flows - BL KM" + NAME

postproc_dir = Path.cwd() / "Research_Runs" / "Postprocessing"
sys.path.append(str(postproc_dir))

# Run installed capacity visualization
from postprocessing import run_IC
run_IC(
    netcdf_path=netcdf_filename,
    model_yaml_path=yaml_filename,
    output_html_path=output_html_path, 
    html_title=html_title
)

# Generate flow/cost report
from flows_cost_util import generate_report
generate_report(
    nc_file_path=netcdf_filename,
    processed_csv_path=flow_csv_filename,
    output_html_file=analysis_html_filename
)

# Run energy flow visualization
# from map import run_energy_flow_visualization
# run_energy_flow_visualization(
#     title=map_title,
#     netcdf_path=netcdf_filename,
#     flow_csv_path=flow_csv_filename,
#     yaml_path=yaml_filename,
#     geojson_path=geojson_filename,
#     output_html_path=map_output_html
# )


Combined page saved to analysis_results/1.Baseline/Installed_Capacities.html


/home/khasselaar/Thesis/Research_Runs/Postprocessing/flows_cost_util.py:266: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_f["tech_label"] = df_f["technology"].map(TECH_LABELS).fillna(df_f["technology"])
/home/khasselaar/Thesis/Research_Runs/Postprocessing/flows_cost_util.py:553: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



Combined report written to analysis_results/1.Baseline/flows_costs_util.html


Comparison between 4 scenarios

In [ ]:
# Jupyter Cell Script for Calliope Multiscenario Comparison
# Run this cell to load the comparison functions and use them

import sys
from pathlib import Path
from itertools import combinations

# Add the script path to Python path
script_path = Path.cwd() / "Research_Runs" / "Postprocessing"
if str(script_path) not in sys.path:
    sys.path.append(str(script_path))

# Import the comparison functions
from calliope_comparison_script import compare_two_runs, detailed_capacity_comparison

def quick_compare_multi(netcdf_paths, yaml_paths, run_names=None, save_csv=True):
    """
    Multiscenario comparison function for Jupyter notebook use.
    
    Parameters
    ----------
    netcdf_paths : list of str or Path
        Paths to the NetCDF output files for each run.
    yaml_paths : list of str or Path
        Paths to the corresponding YAML config files for each run.
    run_names : list of str, optional
        Human-readable names for each run. If None, names are auto-generated as Run1, Run2, ...
    save_csv : bool, default True
        Whether to save each comparison result to CSV files.
    
    Returns
    -------
    results : dict
        Nested dict with keys (i, j) → (comparison_df, detailed_df) for each pair of runs.
    """
    
    n_runs = len(netcdf_paths)
    if len(yaml_paths) != n_runs:
        raise ValueError("`netcdf_paths` and `yaml_paths` must have the same length")
    
    if run_names is None:
        run_names = [f"Run{idx+1}" for idx in range(n_runs)]
    elif len(run_names) != n_runs:
        raise ValueError("`run_names` must have the same length as `netcdf_paths`")
    
    # Prepare output directory
    output_dir = Path.cwd() / "analysis_results" / "1.Baseline" 
    output_dir.mkdir(parents=True, exist_ok=True)
    
    results = {}
    
    # Loop over all unique pairs of runs
    for (i, j) in combinations(range(n_runs), 2):
        name_i = run_names[i]
        name_j = run_names[j]
        print(f"🔄 Comparing {name_i} vs {name_j}")
        print("=" * 60)
        
        # Summary comparison
        comp_df = compare_two_runs(
            netcdf_paths[i], yaml_paths[i],
            netcdf_paths[j], yaml_paths[j],
            run_1_name=name_i, run_2_name=name_j
        )
        
        # Detailed capacity comparison
        det_df = detailed_capacity_comparison(
            netcdf_paths[i], yaml_paths[i],
            netcdf_paths[j], yaml_paths[j],
            run_1_name=name_i, run_2_name=name_j
        )
        
        # Save to CSV if requested
        if save_csv:
            comp_file = output_dir / f"comparison_{name_i}_{name_j}.csv"
            det_file  = output_dir / f"detailed_capacity_{name_i}_{name_j}.csv"
            comp_df.to_csv(comp_file, index=False)
            det_df .to_csv(det_file,  index=False)
            print(f"📁 Saved:")
            print(f"   - {comp_file}")
            print(f"   - {det_file}")
        
        results[(name_i, name_j)] = (comp_df, det_df)
        print()
    
    return results

print("✅ Calliope multirun comparison functions loaded successfully!")


In [ ]:
# Run 1:
SCENARIO1 = "base_run_EV"
TIMESTAMP1 = "0923_2325"

# Run 2:
SCENARIO2 = "base_run_GB"
TIMESTAMP2 = "0923_2325"

# Run 3:
SCENARIO3 = "base_run_HA" 
TIMESTAMP3 = "0923_2325"

# Run 4:
SCENARIO4 = "base_run_KM"
TIMESTAMP4 = "0923_2325"




###########################################################################
NAME1 = "BL_" + SCENARIO1.split("_")[-1] + "_2025" + TIMESTAMP1
NAME2 = "BL_" + SCENARIO2.split("_")[-1] + "_2025" + TIMESTAMP2
NAME3 = "BL_" + SCENARIO3.split("_")[-1] + "_2025" + TIMESTAMP3
NAME4 = "BL_" + SCENARIO4.split("_")[-1] + "_2025" + TIMESTAMP4

netcdf_files = [
    "output/1.Baseline/" + NAME1 + ".nc",
    "output/1.Baseline/" + NAME2 + ".nc",
    "output/1.Baseline/" + NAME3 + ".nc",
    "output/1.Baseline/" + NAME4 + ".nc",
]
yaml_files = [
    "output/1.Baseline/" + NAME1 + ".yaml",
    "output/1.Baseline/" + NAME2 + ".yaml",
    "output/1.Baseline/" + NAME3 + ".yaml",
    "output/1.Baseline/" + NAME4 + ".yaml",
]
run_labels = [NAME1, NAME2, NAME3, NAME4]

# Execute multiscenario comparison
results = quick_compare_multi(
    netcdf_paths=netcdf_files,
    yaml_paths=yaml_files,
    run_names=run_labels,
    save_csv=True
)
